In [ ]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from torch.utils.data import Dataset, DataLoader 
import joblib 
from tqdm import tqdm 


class DeepResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class TinyBlobNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
       
        self.layer1 = DeepResidualBlock(16, 16)            
        self.layer2 = DeepResidualBlock(16, 16)  
        # self.layer3 = DeepResidualBlock(48, 64, stride=2)  # 5x5
        
        self.gap = nn.AdaptiveAvgPool2d(1) 

        self.regressor = nn.Sequential(
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Dropout(0.3), 
            nn.Linear(16, 3) 
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)

        # x = self.layer3(x)
        return self.regressor(self.gap(x).flatten(1))


class BlobDataset(Dataset):
    def __init__(self, noisy_file, clean_file, device):
        clean_dat = joblib.load(clean_file)
        noisy_dat = joblib.load(noisy_file)
        self.noisy = torch.from_numpy(noisy_dat).float().reshape(-1, 1, 20, 20).to(device)
        self.clean = torch.from_numpy(clean_dat["blobs"]).float().reshape(-1, 1, 20, 20).to(device)
        self.targets = torch.cat([
            torch.from_numpy(clean_dat["integrals"]).float().unsqueeze(1),
            torch.from_numpy(clean_dat["cents"]).float()
        ], dim=1).to(device)

    def __len__(self): return len(self.clean)
    def __getitem__(self, idx): return self.noisy[idx], self.clean[idx], self.targets[idx]


DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
NOISE_AMP, NOISE_OFFSET = 0.0005, 1.0
WEIGHTS = torch.tensor([10.0, 1.0, 1.0]).to(DEVICE)

def weighted_mse_loss(input, target, weights):
    return ((input - target)**2 * weights).mean()

train_ds = BlobDataset("TRA_NOISY_DAT.joblib", "TRA_CLEAN_DAT.joblib", DEVICE)
val_ds = BlobDataset("VAL_NOISY_DAT.joblib", "VAL_CLEAN_DAT.joblib", DEVICE)
train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1024, shuffle=False)

model = TinyBlobNet().to(DEVICE)
# Higher weight decay (5e-3) is a strong anti-overfitting measure
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

best_val = float('inf')


for epoch in range(50):
    model.train()
    t_loss = 0.0
    
    for _, clean_orig, targets_orig in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        optimizer.zero_grad(set_to_none=True)
        
        clean, targets = clean_orig.clone(), targets_orig.clone()

        with torch.no_grad():
          
            if torch.rand(1) > 0.5:
                clean = torch.flip(clean, dims=[3]); targets[:, 1] = 19.0 - targets[:, 1]
            if torch.rand(1) > 0.5:
                clean = torch.flip(clean, dims=[2]); targets[:, 2] = 19.0 - targets[:, 2]
            
            sx, sy = torch.randint(-2, 3, (1,)).item(), torch.randint(-2, 3, (1,)).item()
            clean = torch.roll(clean, shifts=(sy, sx), dims=(2, 3))
            targets[:, 1] += sx; targets[:, 2] += sy
            targets[:, 1:] = torch.clamp(targets[:, 1:], 0.0, 19.0)

       
        noise = (2.0 * torch.rand(clean.shape, device=DEVICE) - 1.0).mul_(NOISE_AMP)
        off_val = torch.rand((clean.shape[0], 1), device=DEVICE).mul_(NOISE_OFFSET)
        offset = torch.ones_like(clean, device=DEVICE) * off_val.view(-1, 1, 1, 1)
        noisy_inputs = clean + noise + offset
        
        
        preds = model(noisy_inputs)
        loss = weighted_mse_loss(preds, targets, WEIGHTS)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item()

   
    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for noisy, _, targets in val_loader:
            v_loss += weighted_mse_loss(model(noisy), targets, WEIGHTS).item()

    avg_t, avg_v = t_loss/len(train_loader), v_loss/len(val_loader)
    scheduler.step(avg_v)
    
    print(f"Epoch {epoch+1:02d} | Train: {avg_t:.6f} | Val: {avg_v:.6f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if avg_v < best_val:
        best_val = avg_v
        torch.save(model.state_dict(), "best_tiny_model.pt")